# Ask a Strands Decider endpoint on Amazon SageMaker

This notebook calls a strands decider model that runs on a SageMaker real-time endpoint. You create the
endpoint first with the CDK stack in this folder ([../README.md](../README.md)).

A strands decider model is a **decision model**. It does not generate text. You give it a piece of text, the
**state**, and one or more typed **questions** about that text. It answers each question with a probability.

| You ask | You get |
| --- | --- |
| A yes/no question (`noul`, the API's name for this type) | The probability that the answer is yes |
| A choice between named options (`choice`) | The selected option, a probability for each option, and a confidence |
| A rating on an ordered scale (`score`) | The expected position on the scale, a probability for each level, and a confidence |

Because each answer is a number, your code can branch on it: act on a confident answer, and send an uncertain
one to a person.

In this notebook you:

1. [Connect to the endpoint](#1.-Connect-to-the-endpoint).
2. [Learn the request and the response format](#2.-The-request-and-the-response-format).
3. [Ask each question type](#3.-Ask-each-question-type), then all three in one request.
4. [Use the confidence to decide what to do](#4.-Use-the-confidence-to-decide-what-to-do).
5. [Check an agent's tool call](#5.-Check-an-agent's-tool-call) before it runs.
6. [Handle a request that is not valid](#6.-Handle-a-request-that-is-not-valid).
7. [Delete the endpoint](#7.-Delete-the-endpoint).

**Prerequisites:**

- An endpoint from `npx cdk deploy` ([../README.md](../README.md)). The default is the 2B model,
  `strands-decider-2b-qwen3-5-v1-2610`, on `ml.g6.xlarge`. Any other model works too: the request and the
  response are the same for all of them.
- Python 3.10 or later with `boto3`, and AWS credentials for the account and the Region of the endpoint. The
  notebook runs on your own computer or in SageMaker Studio; it does not need a GPU.
- IAM permission `sagemaker:InvokeEndpoint` and `sagemaker:DescribeEndpoint` on the endpoint.

## 1. Connect to the endpoint

Set `endpoint_name` to the name that `cdk deploy` printed (`EndpointName` in the stack outputs). The Region
comes from your AWS configuration; set `region` to override it.

In [ ]:
import json

import boto3

endpoint_name = "strands-decider-2b-qwen3-5-v1-2610"
region = None  # None: use the Region from your AWS configuration

sagemaker = boto3.client("sagemaker", region_name=region)
runtime = boto3.client("sagemaker-runtime", region_name=region)

status = sagemaker.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
print(f"{endpoint_name}: {status}")

The status must be `InService`. If it is `Creating` or `Updating`, wait and run the cell again.

## 2. The request and the response format

Each request is one JSON object:

| Field | Type | Meaning |
| --- | --- | --- |
| `state` | string | The text to judge: a message, a conversation, a document, an agent's proposed action. |
| `questions` | object | One or more questions about the same `state`. Each key is a name that you choose; the answer comes back under the same name. |
| `questions.<name>.type` | `"noul"`, `"choice"` or `"score"` | The question type. |
| `questions.<name>.instructions` | string | The question, in plain language. |
| `questions.<name>.criteria` | object or list | `choice`: an object that maps each option name to a description (2 or more options). `score`: a list of levels from low to high (2 to 10 levels). `noul`: optional, an object with a `"true"` and a `"false"` description that make the boundary sharper. |

The response holds one answer for each question:

| Question type | Answer fields |
| --- | --- |
| `noul` | `noul`: the probability that the answer is yes, from 0 to 1. |
| `choice` | `choice`: the selected option. `probabilities`: a probability for each option (they add up to 1). `confidence`: from 0 (all options equally likely) to 1 (one option certain). |
| `score` | `score`: the expected level, from 0 (the first level) to the number of levels minus 1. `legend`: the level that each index means. `probabilities`: a probability for each level. `confidence`: from 0 to 1; high when the probability is concentrated on neighbouring levels. |

The response also holds `model` (the model name), `usage` (the number of input tokens, and one output token for
each question, which is the answer) and `latency_ms` (the time that the server took, without the network).

The two helper functions below send a request and print the answers in a short form.

In [ ]:
def ask(body):
    """Send one request to the endpoint and return the parsed JSON response."""
    result = runtime.invoke_endpoint(
        EndpointName=endpoint_name, ContentType="application/json", Body=json.dumps(body)
    )
    return json.loads(result["Body"].read())


def show(response):
    """Print each answer on one line."""
    for name, answer in response["answers"].items():
        if answer["type"] == "noul":
            print(f"{name}: P(yes) = {answer['noul']:.3f}")
        elif answer["type"] == "choice":
            print(
                f"{name}: {answer['choice']} (confidence {answer['confidence']:.2f}), "
                f"probabilities {answer['probabilities']}"
            )
        else:
            nearest = answer["legend"][str(round(answer["score"]))]
            print(
                f"{name}: score {answer['score']:.2f} of 0..{len(answer['legend']) - 1}, "
                f"nearest level {nearest!r} (confidence {answer['confidence']:.2f})"
            )

## 3. Ask each question type

### A yes/no question (`noul`)

Is a support message urgent? The answer is the probability of "yes".

In [ ]:
response = ask(
    {
        "state": "Help! Our payouts have failed for 3 days and our staff cannot get paid on Friday.",
        "questions": {
            "is_urgent": {"type": "noul", "instructions": "Does this message convey urgency?"},
        },
    }
)
show(response)

### A choice (`choice`)

Which team should handle a ticket? The descriptions in `criteria` tell the model what each option means.

In [ ]:
departments = {
    "billing": "Charges, invoices, refunds and payments.",
    "technical": "Bugs, errors and features that do not work.",
    "sales": "Pricing, plans and new purchases.",
}
response = ask(
    {
        "state": "I was charged twice for my October invoice. Please refund the duplicate charge.",
        "questions": {
            "department": {
                "type": "choice",
                "instructions": "Which team should handle this support ticket?",
                "criteria": departments,
            },
        },
    }
)
show(response)

### A rating (`score`)

How frustrated is the customer, on a scale from "Calm" to "Very angry"? The list in `criteria` goes from low
to high. The `score` is an expected value, so it can fall between two levels.

In [ ]:
frustration_levels = ["Calm", "Frustrated", "Very angry"]
response = ask(
    {
        "state": "This is the third time I am writing about this. Nobody has answered, "
        "it is unacceptable, and I am cancelling my account today!",
        "questions": {
            "frustration": {
                "type": "score",
                "instructions": "How frustrated is the customer?",
                "criteria": frustration_levels,
            },
        },
    }
)
show(response)

The score is above the middle of the scale, so the direction is correct, but it is not near "Very angry" and
the confidence is low. `score` answers tend to stay near the middle of the scale, with a lower confidence than
`noul` and `choice` answers. Section 4 shows how to handle a low-confidence answer.

### Three questions in one request

You can ask many questions about the same state in one request. Each answer comes back under the name that you
gave the question.

In [ ]:
triage_questions = {
    "is_urgent": {"type": "noul", "instructions": "Does this message convey urgency?"},
    "department": {
        "type": "choice",
        "instructions": "Which team should handle this support ticket?",
        "criteria": departments,
    },
    "frustration": {
        "type": "score",
        "instructions": "How frustrated is the customer?",
        "criteria": frustration_levels,
    },
}
response = ask(
    {
        "state": "The login page shows error 500 since this morning. None of my team can sign in "
        "and we have a client demo in one hour.",
        "questions": triage_questions,
    }
)
show(response)
print(json.dumps(response, indent=2))

## 4. Use the confidence to decide what to do

Each answer carries its own uncertainty, so your code can decide how much to trust it. On short classification
tasks that the model did not see in training, answers with a confidence of 0.9 or more are right about 95% of
the time ([evaluation results](../../../evaluation/results.md#summary)).

The function below is an example policy, not a recommendation: act automatically on a confident answer, ask the
person who sent the request to confirm a less confident one, and send an uncertain one to a human reviewer.
Choose your own thresholds and test them on your own data. For a `noul` answer, the confidence is `|2p - 1|`:
0 at p = 0.5, 1 at p = 0 or p = 1.

In [ ]:
def next_step(answer, act_at=0.9, confirm_at=0.5):
    """Example policy: turn an answer's confidence into an action."""
    if answer["type"] == "noul":
        confidence = abs(2 * answer["noul"] - 1)
    else:
        confidence = answer["confidence"]
    if confidence >= act_at:
        return "act automatically"
    if confidence >= confirm_at:
        return "ask the sender to confirm"
    return "send to a human reviewer"


for name, answer in response["answers"].items():
    print(f"{name}: {next_step(answer)}")

## 5. Check an agent's tool call

A decision model is useful inside an AI agent, at the points where the agent must make a judgement. In this
example an agent wants to call a weather tool with the argument "Seattle". Two yes/no questions check the
proposed call before it runs: are the arguments grounded in what the user said, and is it too early to call the
tool?

The code runs the check twice. First, the user did not name a city and the agent guessed "Seattle": expect
"ask the user first". Second, the user named Seattle: expect "run the tool call".

The [Strands Agents example](../../../examples/strands/) runs the same check in a live agent, as a hook that
runs before each tool call.

In [ ]:
tool_call_questions = {
    "args_grounded": {
        "type": "noul",
        "instructions": "Are the tool's argument values grounded in facts the user actually provided?",
        "criteria": {
            "true": "every argument value traces back to something the user said",
            "false": "an argument value was guessed or invented, not stated by the user",
        },
    },
    "premature": {
        "type": "noul",
        "instructions": "Is it premature to call this tool now, before clarifying with the user?",
        "criteria": {
            "true": "the assistant should ask a clarifying question before calling the tool",
            "false": "there is nothing left to clarify; calling now is appropriate",
        },
    },
}


def tool_call_state(user_message):
    return (
        "A conversation between a user and an AI assistant is below, followed by a tool call "
        "the assistant now wants to make.\n\n"
        f"--- CONVERSATION ---\nuser: {user_message}\n\n"
        '--- PROPOSED TOOL CALL ---\ntool: get_weather\narguments: {"location": "Seattle"}'
    )


for user_message in ["What is the weather?", "What is the weather in Seattle today?"]:
    body = {"state": tool_call_state(user_message), "questions": tool_call_questions}
    answers = ask(body)["answers"]
    grounded, premature = answers["args_grounded"]["noul"], answers["premature"]["noul"]
    decision = "run the tool call" if grounded >= 0.5 and premature < 0.5 else "ask the user first"
    print(
        f"{user_message!r}: P(grounded) = {grounded:.2f}, P(premature) = {premature:.2f} -> {decision}"
    )

The decision itself is ordinary Python: the model answers the questions, and your code decides what to do with
the answers.

## 6. Handle a request that is not valid

If a request is not valid, the server returns HTTP 422 with an explanation. Through SageMaker, boto3 raises
`ModelError`; the original status code and the server's message are in the error response. This request has no
questions:

In [ ]:
try:
    ask({"state": "Any text.", "questions": {}})
except runtime.exceptions.ModelError as err:
    print("status:", err.response["OriginalStatusCode"])
    print("message:", err.response["OriginalMessage"])

A `state` that is longer than the model's context window (4,096 tokens for the 2B model) is cut to fit by
default. To refuse it with HTTP 422 instead, deploy with `-c serve_args="--strict-window"`
([../README.md](../README.md#configuration)).

## 7. Delete the endpoint

The endpoint costs money for every hour that it runs, whether or not it gets requests. This notebook did not
create the endpoint, so it does not delete it. When you no longer need the endpoint, run this command in
`deploy/sagemaker`:

```bash
npx cdk destroy                    # the default model
npx cdk destroy -c model=<Hub id>  # any other model that you deployed
```